# Steady-state datasets

**Part 2 · From a real grid to power flow samples**

Read a real transmission grid with PowSyBl, move it between exchange formats, and generate perturbed power flow samples solved by OpenLoadFlow.

Rendered version with every output: [https://gfm6w.github.io/02-steady-state.html](https://gfm6w.github.io/02-steady-state.html)

20 min · Part 2 of 4 · pypowsybl · OpenLoadFlow · gridfm-datakit

#### In this part

- Read a real transmission grid with PowSyBl and move it between exchange formats.
- Configure GridFM-Datakit to perturb load and topology and solve every sample with OpenLoadFlow.
- Explore the dataset and see why it contains operating points outside the limits.

In [ ]:
import sys

sys.path.insert(0, "..")
from helpers import MUTED, PALETTE, area_diagram, setup

setup()

In [ ]:
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypowsybl as pp
import yaml

work = Path("outputs/part2")
shutil.rmtree(work, ignore_errors=True)
work.mkdir(parents=True)

## A real grid

The **PEGASE** project mapped the European high-voltage grid for state estimation and security studies. Its 89-bus case is a slice of that grid, published with MATPOWER and the PGLib benchmark library. GridFM-Datakit downloads PGLib cases on demand, and its PowSyBl reader turns the MATPOWER file into a pypowsybl network.

In [ ]:
from gridfm_datakit.network import get_pglib_source_path
from gridfm_datakit.powsybl import load_net

grid = load_net(get_pglib_source_path("case89_pegase")).pp_net
print(len(grid.get_buses()), "buses,", len(grid.get_generators()), "generators,",
      len(grid.get_lines()), "lines,", len(grid.get_2_windings_transformers()), "transformers")
grid.get_voltage_levels().nominal_v.value_counts().rename("voltage levels").to_frame()

Three voltage levels, from 380 kV down to 150 kV. A load flow with OpenLoadFlow checks the case before anything else.

In [ ]:
result = pp.loadflow.run_ac(grid, pp.loadflow.Parameters(distributed_slack=False))
print(result[0].status_text, "in", result[0].iteration_count, "iterations")

buses = grid.get_buses().join(grid.get_voltage_levels().nominal_v, on="voltage_level_id")
ax = (buses.v_mag / buses.nominal_v).plot.hist(bins=30, color=PALETTE[0], edgecolor="white")
ax.set(title="Voltage magnitude, base case", xlabel="pu", ylabel="buses");

The network area diagram around the most loaded substation shows the flows.

In [ ]:
busiest = grid.get_loads().groupby("voltage_level_id").p0.sum().idxmax()
area_diagram(grid, busiest, depth=2, spread=0.5, height=640)

## Exchange formats

European operators exchange grid models as **CGMES**, the Common Grid Model Exchange Standard. PowSyBl writes and reads it, as well as its own **XIIDM** format. A round trip shows what each one keeps.

In [ ]:
grid.save("outputs/part2/pegase89.zip", format="CGMES")
grid.save("outputs/part2/pegase89.xiidm", format="XIIDM")

copies = {
    "MATPOWER source": grid,
    "CGMES 2.4.15": pp.network.load("outputs/part2/pegase89.zip"),
    "XIIDM": pp.network.load("outputs/part2/pegase89.xiidm"),
}
pd.DataFrame(
    {
        name: [len(n.get_buses()), len(n.get_generators()), len(n.get_branches()),
               len(n.get_operational_limits())]
        for name, n in copies.items()
    },
    index=["buses", "generators", "branches", "operational limits"],
)

The topology survives both. The thermal ratings of a MATPOWER case are apparent power limits, which CGMES 2.4.15 cannot express, so they are lost there. XIIDM keeps everything, so that is the file GridFM-Datakit gets. A CGMES file from an operator, whose limits are currents, goes in the same way.

## Configure GridFM-Datakit

For every **load scenario**, datakit solves an AC optimal power flow with PowerModels.jl to get a realistic dispatch. It then draws **topology variants**, here one line or generator out at a time, and solves each one with **OpenLoadFlow** from the same set-points. The set-points are not re-optimised after the outage, so some samples break voltage or thermal limits. That is deliberate. A model meant for operations has to see stressed states too.

The configuration is the YAML file you would pass to `gridfm_datakit generate`.

In [ ]:
config = yaml.safe_load("""
network:
  name: pegase89
  source: file
  reader: powsybl            # read the grid with PowSyBl
  file: outputs/part2/pegase89.xiidm

load:
  generator: agg_load_profile
  agg_profile: default
  scenarios: 40
  sigma: 0.2
  change_reactive_power: true
  global_range: 0.4
  max_scaling_factor: 4.0
  step_size: 0.1
  start_scaling_factor: 1.0

topology_perturbation:
  type: random
  k: 1                       # one element out per variant
  n_topology_variants: 3
  elements: [branch, gen]

generation_perturbation:
  type: none
admittance_perturbation:
  type: none

settings:
  mode: pf                   # keep samples that violate limits
  pf_solver: powsybl         # solve them with OpenLoadFlow
  data_dir: outputs/part2/data
  num_processes: 4
  large_chunk_size: 40
  overwrite: true
  include_dc_res: false
  enable_solver_logs: false
  pf_fast: true
  dcpf_fast: true
  max_iter: 200
  seed: 42
""")

> **Why the costs are flat**
>
> CGMES and the other PowSyBl formats carry no generator costs, so datakit gives every unit the same linear cost. The dispatch still respects every limit of the network, which is what matters for a power flow dataset.

## Generate

In [ ]:
from gridfm_datakit.generate import generate_power_flow_data

generate_power_flow_data(config)

The run prints its progress, then writes everything under `outputs/part2/data/pegase89/raw`.

## Inside the dataset

Every table is keyed by `scenario`, with one row per bus, generator or branch and per sample.

In [ ]:
raw = Path("outputs/part2/data/pegase89/raw")
bus = pd.read_parquet(raw / "bus_data.parquet")
branch = pd.read_parquet(raw / "branch_data.parquet")
print(bus.scenario.nunique(), "samples from", bus.load_scenario_idx.nunique(), "load scenarios")
bus.head()

Each load scenario is a point on a yearly load curve, with local noise on every bus.

In [ ]:
demand = bus.groupby("load_scenario_idx").Pd.sum()
ax = demand.plot(marker="o", markersize=4, color=PALETTE[0])
ax.set(title="Total demand per load scenario", xlabel="load scenario", ylabel="MW");

The outages spread the voltages and push some branches over their rating.

In [ ]:
branch["loading"] = np.hypot(branch.pf, branch.qf) / branch.rate_a
in_service = branch[(branch.br_status == 1) & (branch.rate_a > 0)]

fig, (ax_v, ax_l) = plt.subplots(1, 2, figsize=(10, 3.6))
ax_v.hist(bus.Vm, bins=40, color=PALETTE[0], edgecolor="white")
ax_v.axvline(bus.min_vm_pu.iloc[0], color=MUTED, linestyle="--", linewidth=1)
ax_v.axvline(bus.max_vm_pu.iloc[0], color=MUTED, linestyle="--", linewidth=1)
ax_v.set(title="Voltage magnitude, all samples", xlabel="pu", ylabel="bus samples")
ax_l.hist(in_service.loading, bins=40, color=PALETTE[1], edgecolor="white")
ax_l.axvline(1.0, color=MUTED, linestyle="--", linewidth=1)
ax_l.set(title="Branch loading, all samples", xlabel="|S| / rating", ylabel="branch samples")
fig.tight_layout()

overloaded = in_service.groupby("scenario").loading.max().gt(1.0).mean()
print(f"{overloaded:.0%} of samples have at least one overloaded branch")

## Your turn

> **✎ Exercise · Lines only, more variants**
>
> Restrict the outages to branches and draw five topology variants per load scenario. Save the configuration as `outputs/part2/lines_only.yaml` and run it with the command line, `gridfm_datakit generate outputs/part2/lines_only.yaml`, from a terminal or from Python with `subprocess`. Then compare the share of samples with an overloaded branch.

In [ ]:
# Your turn


<details><summary><b>Show a solution</b></summary>

```python
lines_only = {
    **config,
    "topology_perturbation": {
        "type": "random",
        "k": 1,
        "n_topology_variants": 5,
        "elements": ["branch"],
    },
    "settings": {**config["settings"], "data_dir": "outputs/part2/lines_only"},
}
Path("outputs/part2/lines_only.yaml").write_text(yaml.safe_dump(lines_only))

import subprocess

command = [sys.executable, "-m", "gridfm_datakit.cli", "generate", "outputs/part2/lines_only.yaml"]
subprocess.run(command, check=True, capture_output=True)

b2 = pd.read_parquet("outputs/part2/lines_only/pegase89/raw/branch_data.parquet")
b2 = b2[(b2.br_status == 1) & (b2.rate_a > 0)]
b2["loading"] = np.hypot(b2.pf, b2.qf) / b2.rate_a
print(f"{b2.groupby('scenario').loading.max().gt(1.0).mean():.0%} of samples overloaded")
```

</details>

> **Next**
>
> [Part 3](03-dynamic-initial-state.ipynb) adds time. Each sample becomes a Dynawo trajectory that starts from a different operating point.